# ZenteiQ AI Tech Innovations — Task 2: Dense Model (Qwen)
## MaxText Qwen3-1.7B TPU Training Run & Benchmark

This notebook provides a detailed setup and execution workflow for training the **Qwen3-1.7** base model on a **TPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

We will clone the MaxText repository and install its dependencies. When running on a TPU-only environment (such as Google Colab with the TPU runtime selected). We will install the dependencies using `uv` for fast installations.

In [1]:
!tpu-info

Libtpu version: 0.0.21                                                          
Accelerator type: v5e                                                           
                                                                                
TPU Chips                                      
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━┓
┃ Chip        ┃ Type         ┃ Devices ┃ PID  ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━┩
│ /dev/vfio/0 │ TPU v5e chip │ 1       │ None │
└─────────────┴──────────────┴─────────┴──────┘
╭───────────────────────── Runtime Utilization Status ─────────────────────────╮
│ WARNING: Libtpu metrics unavailable. Is there a framework using the TPU? See │
│ ]8;id=3637870;https://github.com/google/cloud-accelerator-diagnostics/tree/main/tpu_info\tpu_info docs]8;;\ for more information.                                          │
╰──────────────────────────────────────────────────────────────────────────────╯
TPU Runtime Utilization                
┏━━━━━━

In [2]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Install MaxText TPU dependencies
!uv pip install -e .[tpu]

# !uv pip install maxtext[tpu]==0.2.3 --resolution=lowest

# Restart the runtime after this step if run in Colab!

Cloning into 'maxtext'...
remote: Enumerating objects: 97124, done.
remote: Counting objects: 100% (2206/2206), done.
remote: Compressing objects: 100% (1131/1131), done.
remote: Total 97124 (delta 1794), reused 1089 (delta 1075), pack-reused 94918 (from 5)
Receiving objects: 100% (97124/97124), 411.68 MiB | 44.80 MiB/s, done.
Resolving deltas: 100% (70829/70829), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 107.7 MB/s eta 0:00:0000:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 235 packages in 2.56s                                       
Prepared 116 packages in 29.19s                                          
Uninstalled 19 packages in 2.38s
Installed 118 packages in 416ms                             
 - absl-py==1.4.0
 + absl-py==2.4.0
 + antlr4-python3-runtime==4.9.3
 + aqtp==0.9.0
 + astroid==4.0.4
 + astunparse==1.6.3
 + auditwheel==6.7.0
 + black==25.12.0
 + build==1.5.0
 + cfgv==3.5.0
 - chex==0.1.90
 + chex==0.1.92
 + cloud-accele

### 2. Verify JAX TPU Backend Connection

Run the following cell to confirm that JAX successfully detects the TPU backend.

In [3]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX Version: 0.10.2
Available devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Default backend: tpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `qwen3-1.7b` | Specifies the base architecture layout to use. | Loads structural config from `configs/models/qwen3-1.7b.yml` setting embedding dimensions (`base_emb_dim: 1024`), intermediate MLP dimension (`base_mlp_dim: 3072`), attention query/KV heads (`16`/`8`), and number of layers (`28`). |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on TPU memory. This allows pure computational throughput benchmarking. |
| **`run_name`** | `qwen3-1.7b-tpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `qwen3-1.7b-tpu/` for output checkpoints and statistics. |

### 4. Execute TPU Training (Qwen 1.7B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [14]:
!cat src/maxtext/configs/base.yml

# Copyright 2023–2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# This sentinel is a reminder to choose a real run name.
# If there is already a checkpoint under this run, that checkpoint will auto-resume.
#
# NOTE: Some unit/integration tests in MaxText do not always run this file directly.
# When running in decoupled mode (DECOUPLE_GCLOUD=TRUE), tests may use
# `decoupled_base_test.yml` instead of `base.yml` via `tests/utils/test_helpers.py`.
run_name: ""

model_name: "default"

In [4]:
!uv pip uninstall torch -y

Using Python 3.12.13 environment at: /usr
Uninstalled 1 package in 1.24s
 - torch==2.9.0+cpu (from https://download.pytorch.org/whl/cpu/torch-2.9.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl)


In [8]:
# Run training for 50 steps on TPU

!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    hardware=tpu \
    model_name=qwen3-1.7b \
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=$(pwd)/tpu-output-1.7b \
    run_name=qwen3-1.7-tpu \
    dtype=bfloat16 \
    weight_dtype=bfloat16 \
    grad_dtype=bfloat16 \
    per_device_batch_size=4 \
    attention=autoselected \
    max_target_length=2048 \
    enable_checkpointing=false

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
2026-06-18 12:40:23.790295: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-18 12:40:23.830391: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-18 12:40:24.948790: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[tra

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 5. Parse Metrics & Logs

MaxText prints metric logs regularly during training. We will extract the compilation time, step times, loss, learning rate, and TFLOPs using a helper script.

In [19]:
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [18]:
!uv pip install --upgrade setuptools tensorboard


Using Python 3.12.13 environment at: /usr
Resolved 13 packages in 194ms                                        
Prepared 1 package in 6ms                                                
Uninstalled 1 package in 4ms
Installed 1 package in 11ms                                 
 - protobuf==6.33.6
 + protobuf==7.35.1


In [21]:
# %tensorboard --logdir gpu-output-v2/qwen3-0.6b-gpu/tensorboard/


In [3]:
!ls 

drive  maxtext	sample_data  tpu-output


In [10]:
!cp -r tpu-output-1.7b /content/drive/MyDrive/
